# Trading Bot - Produção
Bot de trading automatizado usando o modelo XGBoost

## Resultados do Backtesting (Threshold 0.55)
**Período:** 2021-2026 (5 anos) | **Capital inicial:** $10,000

| Métrica | Valor |
|---------|-------|
| **Retorno Total** | **+1,143.12%** ($10k → $124k) |
| **Sharpe Ratio** | **11.46** (excepcional) |
| **Max Drawdown** | **-1.33%** (muito baixo) |
| **Win Rate** | **70.6%** |
| **Profit Factor** | **3.84** |
| **Trades Totais** | 14,082 |
| **Holding Médio** | 2.1 horas |
| **vs Buy & Hold** | +1,188 pontos (B&H: -45%) |

**Top Performers:**
* SOL/USDT: +2,067% ($24,545)
* AVAX/USDT: +1,486% ($17,652)
* ETH/USDT: +1,199% ($14,234)

## Configuração
1. Definir API keys da exchange (modo dry-run não precisa)
2. Configurar pares a operar
3. Ajustar capital e position size
4. Escolher modo: dry-run (simulação) ou live (real)

In [0]:
# lista todos os "cofres" (Scopes) com acesso
display(dbutils.secrets.listScopes())
# lista todas as chaves dentro do cofre
display(dbutils.secrets.list(scope="trading-bot"))

In [0]:
%pip install ccxt xgboost scikit-learn

In [0]:
import ccxt
import pandas as pd
import numpy as np
import time
import joblib
from datetime import datetime, timedelta
import warnings
import json
import os

warnings.filterwarnings('ignore')
print(f"CCXT version: {ccxt.__version__}")
print(f"Pandas version: {pd.__version__}")

## Configuração

Definir parâmetros de trading, pares, e credenciais da exchange

In [0]:
# configurações do arquivo trading_params.json
CONFIG_PATH = os.path.join(os.path.dirname(os.getcwd()), 'config', 'trading_params.json')
with open(CONFIG_PATH, 'r') as f:
    config = json.load(f)

# modo de operação
DRY_RUN = True  # True = simulação sem capital real, False = trading real

# exchange
EXCHANGE = 'kucoin'
TIMEFRAME = '1h' # timeframe principal (não mudar)

# pares para operar
TRADING_PAIRS = config['symbols']['active']

# capital / gestão de risco
INITIAL_CAPITAL = 10000.0  # USD
POSITION_SIZE = 0.95       # 95% do capital por trade

# THRESHOLD e MAX_POSITIONS agora vêm do arquivo de configuração
MAX_POSITIONS = config['trading']['max_open_positions']
THRESHOLD = config['trading']['min_probability_threshold']

# modelo treinado
MODEL_PATH = os.path.join(os.path.dirname(os.getcwd()), 'models', 'xgboost_atr_target.pkl')

# stop loss e take profit ADAPTATIVOS baseados em ATR
ATR_MULTIPLIER_TP = config['risk_management']['take_profit_atr_multiplier']
ATR_MULTIPLIER_SL = config['risk_management']['stop_loss_atr_multiplier']

# intervalo de verificação
CHECK_INTERVAL = 60  # segundos

try:
    # carrega o databricks secrets
    API_KEY = dbutils.secrets.get(scope="trading-bot", key="api-key")
    API_SECRET = dbutils.secrets.get(scope="trading-bot", key="api-secret")
    
    # kucoin passphrase (opcional)
    try:
        API_PASSPHRASE = dbutils.secrets.get(scope="trading-bot", key="api-passphrase")
    except:
        API_PASSPHRASE = ''  # vazio = tentar sem passphrase
    
    print("Credenciais carregadas do databricks secrets")
    SECRETS_LOADED = True
except Exception as e:
    # se secrets não configurados, usar valores vazios
    API_KEY = ''
    API_SECRET = ''
    API_PASSPHRASE = ''
    SECRETS_LOADED = False
    if not DRY_RUN:
        print("Secrets não configurados!")
    else:
        print("Secrets não configurados (OK para modo DRY RUN)")

# logs
print(f"\n{'='*70}")
print(f"CONFIGURAÇÃO DO BOT")
print(f"{'='*70}")
print(f"\nConfigurações carregadas de: {CONFIG_PATH}")
print(f"\nModo: {'DRY RUN (Simulação)' if DRY_RUN else 'LIVE TRADING (Real)'}")
print(f"\nExchange: {EXCHANGE.upper()}")
print(f"   • Timeframe principal: {TIMEFRAME}")
print(f"   • Timeframes adicionais: 15m, 4h (para features)")
print(f"   • Pares: {len(TRADING_PAIRS)} ativos")
for pair in TRADING_PAIRS:
    print(f"     - {pair}")

print(f"\nCapital:")
print(f"   • Inicial: ${INITIAL_CAPITAL:,.2f}")
print(f"   • Position size: {POSITION_SIZE*100:.0f}%")
print(f"   • Max posições: {MAX_POSITIONS} (config)")

print(f"\nEstratégia:")
print(f"   • Threshold: {THRESHOLD:.2f} (config)")
print(f"   • TP/SL: ADAPTATIVOS baseados em ATR")
print(f"   • Take Profit: close + {ATR_MULTIPLIER_TP}x ATR (config)")
print(f"   • Stop Loss: close - {ATR_MULTIPLIER_SL}x ATR (config)")

print(f"\nAutenticação:")
if SECRETS_LOADED:
    print(f"   • Status: Credenciais carregadas")
    print(f"   • Scope: trading-bot")
    print(f"   • API Key: {'*' * 20} (oculto)")
    if API_PASSPHRASE:
        print(f"   • Passphrase: {'*' * 10} (configurado)")
    else:
        print(f"   • Passphrase: não configurado")
else:
    print(f"   • Status: {'Secrets não configurados' if not DRY_RUN else 'OK para DRY RUN'}")
    if not DRY_RUN:
        print(f"   • BLOQUEIO: Modo LIVE requer secrets configurados")

print(f"\nIntervalo: {CHECK_INTERVAL}s entre ciclos")
print(f"\nModelo: xgboost_atr_target.pkl (~96 features - Lote 1)")
print(f"   • Path: {MODEL_PATH}")


In [0]:
print("carregando modelo...\n")
try:
    model = joblib.load(MODEL_PATH)
    print(f"\nDetalhes do modelo:")
    print(f"   • Tipo: {type(model).__name__}")
    print(f"   • Features esperadas: {model.n_features_in_}")
    
    # verifica feature names
    if hasattr(model, 'feature_names_in_'):
        print(f"\nFeatures do modelo:")
        feature_list = list(model.feature_names_in_)
        
        # agrupa por timeframe
        timeframes = {'1h': [], '15m': [], '4h': [], 'btc_eth': []}
        for feat in feature_list:
            if '_1h' in feat:
                timeframes['1h'].append(feat)
            elif '_15m' in feat:
                timeframes['15m'].append(feat)
            elif '_4h' in feat:
                timeframes['4h'].append(feat)
            else:
                timeframes['btc_eth'].append(feat)
        
        print(f"   • Timeframe 1h: {len(timeframes['1h'])} features")
        print(f"   • Timeframe 15m: {len(timeframes['15m'])} features")
        print(f"   • Timeframe 4h: {len(timeframes['4h'])} features")
        print(f"   • Correlações BTC/ETH: {len(timeframes['btc_eth'])} features")
        print(f"\nTotal: {len(feature_list)} features validadas")
    else:
        print(f"   Feature names não disponíveis no modelo")
        
except FileNotFoundError:
    print(f"ERRO: Modelo não encontrado!")
    print(f"   Path: {MODEL_PATH}")
    print(f"\nVerifique se:")
    print(f"   1. O caminho está correto")
    print(f"   2. O arquivo existe no diretório")
    print(f"   3. O modelo foi treinado e salvo corretamente")
    raise
except Exception as e:
    print(f"ERRO ao carregar modelo: {e}")
    raise

In [0]:
# Inspecionar features BTC/ETH
if hasattr(model, 'feature_names_in_'):
    btc_eth_features = [f for f in model.feature_names_in_ if '_1h' not in f and '_15m' not in f and '_4h' not in f]
    print(f"BTC/ETH features ({len(btc_eth_features)}):")
    for i, feat in enumerate(btc_eth_features, 1):
        print(f"  {i}. {feat}")

## Feature Engineering
Funções para calcular as ~96 features esperadas pelo modelo (Lote 1 completo)

In [0]:
def calculate_timeframe_features(df, suffix):
    """
    Calcula features técnicas para um timeframe específico (Lote 1 completo).
    
    Features Originais (13):
    1. log_return, 2. rsi, 3. volatility, 4. ma_close, 5. ma_volume
    6. z_score_close, 7. z_score_volume, 8. atr, 9. momentum
    10. vol_change, 11-13. bb_upper, bb_lower, bb_position
    
    Novas Features - Lote 1 (13-15 dependendo do timeframe):
    14. vwap, 15. vwap_distance
    16-18. body_range_ratio, upper_shadow_ratio, lower_shadow_ratio
    19-21. roc_5, roc_10, roc_20
    22. obv
    23. adx
    24. vol_ratio
    25-26. resistance_break, support_break
    27-28. hour_sin, hour_cos (apenas para _15m e _1h)
    
    Args:
        df: DataFrame com OHLCV
        suffix: sufixo para as features (ex: '_1h', '_15m', '_4h')
    
    Returns:
        dict com as features + ATR para TP/SL
    """
    df = df.copy()
    
    # ========== FEATURES ORIGINAIS (13) ==========
    
    # 1. log_return
    df['log_return'] = np.log(df['close'] / df['close'].shift(1))
    
    # 2. RSI (14 períodos)
    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['rsi'] = 100 - (100 / (1 + rs))
    
    # 3. Volatility (20 períodos)
    df['volatility'] = df['log_return'].rolling(window=20).std()
    
    # 4. MA Close (20 períodos)
    df['ma_close'] = df['close'].rolling(window=20).mean()
    
    # 5. MA Volume (20 períodos)
    df['ma_volume'] = df['volume'].rolling(window=20).mean()
    
    # 6. Z-Score Close (20 períodos)
    ma_20 = df['close'].rolling(window=20).mean()
    std_20 = df['close'].rolling(window=20).std()
    df['z_score_close'] = (df['close'] - ma_20) / std_20
    
    # 7. Z-Score Volume (20 períodos)
    ma_vol_20 = df['volume'].rolling(window=20).mean()
    std_vol_20 = df['volume'].rolling(window=20).std()
    df['z_score_volume'] = (df['volume'] - ma_vol_20) / std_vol_20
    
    # 8. ATR (14 períodos)
    high_low = df['high'] - df['low']
    high_close = np.abs(df['high'] - df['close'].shift())
    low_close = np.abs(df['low'] - df['close'].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = np.max(ranges, axis=1)
    df['atr'] = true_range.rolling(14).mean()
    
    # 9. Momentum (3 períodos)
    df['momentum'] = (df['close'] - df['close'].shift(3)) / df['close'].shift(3)
    
    # 10. Vol Change
    df['vol_change'] = (df['volume'] - df['volume'].shift(1)) / df['volume'].shift(1)
    
    # 11-13. Bollinger Bands (20, 2)
    bb_ma = df['close'].rolling(window=20).mean()
    bb_std = df['close'].rolling(window=20).std()
    df['bb_upper'] = bb_ma + (2 * bb_std)
    df['bb_lower'] = bb_ma - (2 * bb_std)
    df['bb_position'] = (df['close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])
    
    # ========== LOTE 1: NOVAS FEATURES (13-15) ==========
    
    # 14. VWAP (Volume Weighted Average Price - 20 períodos)
    df['vwap'] = (df['close'] * df['volume']).rolling(window=20).sum() / df['volume'].rolling(window=20).sum()
    
    # 15. VWAP Distance (percentual)
    df['vwap_distance'] = (df['close'] - df['vwap']) / df['vwap']
    
    # 16. Body-to-Range Ratio (candle psychology)
    df['body_range_ratio'] = np.abs(df['close'] - df['open']) / (df['high'] - df['low'])
    
    # 17-18. Shadow Ratios (candle psychology)
    upper_shadow = df['high'] - df[['open', 'close']].max(axis=1)
    lower_shadow = df[['open', 'close']].min(axis=1) - df['low']
    df['upper_shadow_ratio'] = upper_shadow / (df['high'] - df['low'])
    df['lower_shadow_ratio'] = lower_shadow / (df['high'] - df['low'])
    
    # 19-21. ROC (Rate of Change) múltiplos períodos
    df['roc_5'] = (df['close'] - df['close'].shift(5)) / df['close'].shift(5)
    df['roc_10'] = (df['close'] - df['close'].shift(10)) / df['close'].shift(10)
    df['roc_20'] = (df['close'] - df['close'].shift(20)) / df['close'].shift(20)
    
    # 22. OBV (On-Balance Volume - 20 períodos)
    obv_delta = np.where(df['close'] > df['close'].shift(1), df['volume'],
                         np.where(df['close'] < df['close'].shift(1), -df['volume'], 0))
    df['obv'] = pd.Series(obv_delta, index=df.index).rolling(window=20).sum()
    
    # 23. ADX (Average Directional Index - 14 períodos)
    high_diff = df['high'] - df['high'].shift(1)
    low_diff = df['low'].shift(1) - df['low']
    plus_dm = np.where((high_diff > low_diff) & (high_diff > 0), high_diff, 0)
    minus_dm = np.where((low_diff > high_diff) & (low_diff > 0), low_diff, 0)
    plus_di = (pd.Series(plus_dm, index=df.index).rolling(14).mean() * 100) / df['atr']
    minus_di = (pd.Series(minus_dm, index=df.index).rolling(14).mean() * 100) / df['atr']
    dx = np.abs(plus_di - minus_di) * 100 / (plus_di + minus_di)
    df['adx'] = dx.rolling(14).mean()
    
    # 24. Volatility Ratio (curto prazo / longo prazo)
    volatility_short = df['log_return'].rolling(window=10).std()
    volatility_long = df['log_return'].rolling(window=50).std()
    df['vol_ratio'] = volatility_short / volatility_long
    
    # 25. Resistance Break (distância da máxima de 20 períodos)
    resistance_20 = df['high'].rolling(window=20).max()
    df['resistance_break'] = (df['close'] - resistance_20) / resistance_20
    
    # 26. Support Break (distância da mínima de 20 períodos)
    support_20 = df['low'].rolling(window=20).min()
    df['support_break'] = (df['close'] - support_20) / support_20
    
    # 27-28. Hour of Day (cíclico - apenas para timeframes com granularidade horária)
    if suffix in ['_15m', '_1h']:
        df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
        df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)
    
    # Selecionar última linha e criar dicionário de features
    last_row = df.iloc[-1]
    
    features = {
        # Originais (13)
        f'log_return{suffix}': last_row['log_return'],
        f'rsi{suffix}': last_row['rsi'],
        f'volatility{suffix}': last_row['volatility'],
        f'ma_close{suffix}': last_row['ma_close'],
        f'ma_volume{suffix}': last_row['ma_volume'],
        f'z_score_close{suffix}': last_row['z_score_close'],
        f'z_score_volume{suffix}': last_row['z_score_volume'],
        f'atr{suffix}': last_row['atr'],
        f'momentum{suffix}': last_row['momentum'],
        f'vol_change{suffix}': last_row['vol_change'],
        f'bb_upper{suffix}': last_row['bb_upper'],
        f'bb_lower{suffix}': last_row['bb_lower'],
        f'bb_position{suffix}': last_row['bb_position'],
        
        # Lote 1 (13-15)
        f'vwap{suffix}': last_row['vwap'],
        f'vwap_distance{suffix}': last_row['vwap_distance'],
        f'body_range_ratio{suffix}': last_row['body_range_ratio'],
        f'upper_shadow_ratio{suffix}': last_row['upper_shadow_ratio'],
        f'lower_shadow_ratio{suffix}': last_row['lower_shadow_ratio'],
        f'roc_5{suffix}': last_row['roc_5'],
        f'roc_10{suffix}': last_row['roc_10'],
        f'roc_20{suffix}': last_row['roc_20'],
        f'obv{suffix}': last_row['obv'],
        f'adx{suffix}': last_row['adx'],
        f'vol_ratio{suffix}': last_row['vol_ratio'],
        f'resistance_break{suffix}': last_row['resistance_break'],
        f'support_break{suffix}': last_row['support_break']
    }
    
    # Adicionar hour_sin/cos se aplicável
    if suffix in ['_15m', '_1h']:
        features[f'hour_sin{suffix}'] = last_row['hour_sin']
        features[f'hour_cos{suffix}'] = last_row['hour_cos']
    
    return features, last_row['atr']  # retornar ATR para TP/SL


def calculate_btc_eth_features(df_asset, df_btc, df_eth):
    """
    Calcula as 14 features de correlação BTC/ETH.
    
    Features:
    1-4. btc_close, btc_log_return, btc_volume, btc_volatility
    5-8. eth_close, eth_log_return, eth_volume, eth_volatility
    9. relative_strength_btc - força relativa vs BTC
    10. relative_strength_eth - força relativa vs ETH
    11. volatility_ratio_btc - ratio de volatilidade vs BTC
    12. volume_ratio_btc - ratio de volume vs BTC
    13. beta_btc - beta em relação ao BTC
    14. market_breadth - % de ativos (BTC, ETH) com retorno positivo
    """
    # BTC features
    btc_close = df_btc.iloc[-1]['close']
    btc_log_return = np.log(df_btc.iloc[-1]['close'] / df_btc.iloc[-2]['close'])
    btc_volume = df_btc.iloc[-1]['volume']
    btc_volatility = np.log(df_btc['close'] / df_btc['close'].shift(1)).rolling(20).std().iloc[-1]
    
    # ETH features
    eth_close = df_eth.iloc[-1]['close']
    eth_log_return = np.log(df_eth.iloc[-1]['close'] / df_eth.iloc[-2]['close'])
    eth_volume = df_eth.iloc[-1]['volume']
    eth_volatility = np.log(df_eth['close'] / df_eth['close'].shift(1)).rolling(20).std().iloc[-1]
    
    # Asset features
    asset_log_return = np.log(df_asset.iloc[-1]['close'] / df_asset.iloc[-2]['close'])
    asset_volume = df_asset.iloc[-1]['volume']
    asset_volatility = np.log(df_asset['close'] / df_asset['close'].shift(1)).rolling(20).std().iloc[-1]
    
    # Relative strength
    relative_strength_btc = asset_log_return - btc_log_return
    relative_strength_eth = asset_log_return - eth_log_return
    
    # Volatility ratio
    volatility_ratio_btc = asset_volatility / btc_volatility if btc_volatility != 0 else 0
    
    # Volume ratio
    volume_ratio_btc = asset_volume / btc_volume if btc_volume != 0 else 0
    
    # Beta (correlação * (std_asset / std_btc))
    asset_returns = np.log(df_asset['close'] / df_asset['close'].shift(1)).dropna()
    btc_returns = np.log(df_btc['close'] / df_btc['close'].shift(1)).dropna()
    
    # Alinhar tamanhos
    min_len = min(len(asset_returns), len(btc_returns))
    asset_returns = asset_returns.iloc[-min_len:]
    btc_returns = btc_returns.iloc[-min_len:]
    
    correlation = asset_returns.corr(btc_returns)
    beta_btc = correlation * (asset_returns.std() / btc_returns.std()) if btc_returns.std() != 0 else 0
    
    # Market Breadth: % de ativos principais (BTC, ETH) com retorno positivo
    assets_positive = (1 if btc_log_return > 0 else 0) + (1 if eth_log_return > 0 else 0)
    market_breadth = assets_positive / 2.0  # 0.0, 0.5, ou 1.0
    
    features = {
        'btc_close': btc_close,
        'btc_log_return': btc_log_return,
        'btc_volume': btc_volume,
        'btc_volatility': btc_volatility,
        'eth_close': eth_close,
        'eth_log_return': eth_log_return,
        'eth_volume': eth_volume,
        'eth_volatility': eth_volatility,
        'relative_strength_btc': relative_strength_btc,
        'relative_strength_eth': relative_strength_eth,
        'volatility_ratio_btc': volatility_ratio_btc,
        'volume_ratio_btc': volume_ratio_btc,
        'beta_btc': beta_btc,
        'market_breadth': market_breadth
    }
    
    return features


def calculate_features(df_1h, df_15m, df_4h, df_btc_1h, df_btc_15m, df_btc_4h, df_eth_1h):
    """
    Calcula todas as 96 features esperadas pelo modelo XGBoost (Lote 1 completo).
    
    Distribuição:
    - Timeframe 1h: 28 features (13 originais + 15 Lote 1 com hour_sin/cos)
    - Timeframe 15m: 28 features (13 originais + 15 Lote 1 com hour_sin/cos)
    - Timeframe 4h: 26 features (13 originais + 13 Lote 1 sem hour_sin/cos)
    - BTC/ETH correlações: 14 features (includes market_breadth)
    Total: 28 + 28 + 26 + 14 = 96 features
    
    Args:
        df_1h: DataFrame com OHLCV 1h do ativo
        df_15m: DataFrame com OHLCV 15m do ativo
        df_4h: DataFrame com OHLCV 4h do ativo
        df_btc_1h: DataFrame com OHLCV 1h do BTC
        df_btc_15m: DataFrame com OHLCV 15m do BTC
        df_btc_4h: DataFrame com OHLCV 4h do BTC
        df_eth_1h: DataFrame com OHLCV 1h do ETH
    
    Returns:
        Series com as 96 features + ATR para TP/SL
    """
    # Calcular features por timeframe
    features_1h, atr_1h = calculate_timeframe_features(df_1h, '_1h')
    features_15m, _ = calculate_timeframe_features(df_15m, '_15m')
    features_4h, _ = calculate_timeframe_features(df_4h, '_4h')
    
    # Calcular features de correlação BTC/ETH
    features_btc_eth = calculate_btc_eth_features(df_1h, df_btc_1h, df_eth_1h)
    
    # Combinar todas as features na ordem correta
    all_features = {}
    all_features.update(features_1h)
    all_features.update(features_15m)
    all_features.update(features_4h)
    all_features.update(features_btc_eth)
    
    return pd.Series(all_features), atr_1h

print("  • calculate_timeframe_features: 26-28 features por timeframe")
print("    - 13 originais + 13-15 Lote 1 (15 com hour_sin/cos para 1h e 15m)")
print("  • calculate_btc_eth_features: 14 features de correlação (includes market_breadth)")
print("  • calculate_features: 96 features totais")
print("    - 1h: 28 features | 15m: 28 features | 4h: 26 features | BTC/ETH: 14 features")

## Classe principal
processo completo com gestão de posições, risco, e logging

In [0]:
class TradingBot:    
    def __init__(self, exchange_id, model, pairs, threshold, dry_run=True, 
                 api_key='', api_secret='', api_passphrase='', initial_capital=10000):
        self.model = model
        self.pairs = pairs
        self.threshold = threshold
        self.dry_run = dry_run
        self.capital = initial_capital
        self.initial_capital = initial_capital
        
        # inicia exchange
        exchange_class = getattr(ccxt, exchange_id)
        exchange_config = {
            'apiKey': api_key,
            'secret': api_secret,
            'enableRateLimit': True,
            'options': {'defaultType': 'spot'}
        }
        
        # kucoin exige passphrase adicional?
        if exchange_id == 'kucoin' and api_passphrase:
            exchange_config['password'] = api_passphrase
        
        self.exchange = exchange_class(exchange_config)
        
        # tracking de posições
        self.positions = {}  # {symbol: {'entry_price', 'entry_time', 'size', 'tp', 'sl', 'atr'}}
        self.trade_history = []  # lista de trades executados
        
        print(f"Bot inicializado")
        print(f"   • Exchange: {exchange_id.upper()}")
        print(f"   • Modo: {'DRY RUN' if dry_run else 'LIVE'}")
        print(f"   • Pares: {len(pairs)}")
        print(f"   • Threshold: {threshold}")
    
    def fetch_ohlcv(self, symbol, timeframe='1h', limit=100):
        """Busca dados OHLCV da exchange."""
        try:
            ohlcv = self.exchange.fetch_ohlcv(symbol, timeframe, limit=limit)
            df = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
            df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
            df = df.set_index('timestamp')  # Set timestamp as index for time-series operations
            return df
        except Exception as e:
            print(f"Erro ao buscar dados de {symbol} {timeframe}: {e}")
            return None
    
    def get_features(self, symbol):
        """Calcula features para um símbolo."""
        # Buscar dados do ativo em 3 timeframes
        df_1h = self.fetch_ohlcv(symbol, '1h', limit=100)
        df_15m = self.fetch_ohlcv(symbol, '15m', limit=100)
        df_4h = self.fetch_ohlcv(symbol, '4h', limit=100)
        
        # Buscar dados do BTC em 3 timeframes
        df_btc_1h = self.fetch_ohlcv('BTC/USDT', '1h', limit=100)
        df_btc_15m = self.fetch_ohlcv('BTC/USDT', '15m', limit=100)
        df_btc_4h = self.fetch_ohlcv('BTC/USDT', '4h', limit=100)
        
        # Buscar dados do ETH (apenas 1h para correlações)
        df_eth_1h = self.fetch_ohlcv('ETH/USDT', '1h', limit=100)
        
        # Verificar se todos os dados foram obtidos
        if any(df is None for df in [df_1h, df_15m, df_4h, df_btc_1h, df_btc_15m, df_btc_4h, df_eth_1h]):
            return None, None, None
        
        # Calcular features (retorna Series + ATR)
        features, atr = calculate_features(df_1h, df_15m, df_4h, df_btc_1h, df_btc_15m, df_btc_4h, df_eth_1h)
        current_price = df_1h.iloc[-1]['close']
        
        return features, current_price, atr
    
    def predict(self, features):
        """Gera predição do modelo."""
        X = features.values.reshape(1, -1)
        prob = self.model.predict_proba(X)[0][1]
        return prob
    
    def explain_signal(self, features, prob):
        """Explica o motivo do sinal de compra baseado nas features."""
        print(f"\n   📊 ANÁLISE DAS CONDIÇÕES DE MERCADO:")
        print(f"   {'─'*60}")
        print(f"   Probabilidade: {prob*100:.2f}%")
        print(f"")
        
        # Extrair features principais do timeframe 1h (mais importante)
        momentum_1h = features['momentum_1h'] * 100
        rsi_1h = features['rsi_1h']
        volatility_1h = features['volatility_1h']
        bb_position_1h = features['bb_position_1h']
        log_return_1h = features['log_return_1h'] * 100
        vol_change_1h = features['vol_change_1h'] * 100
        
        # Correlações BTC/ETH
        btc_log_return = features['btc_log_return'] * 100
        relative_strength_btc = features['relative_strength_btc'] * 100
        beta_btc = features['beta_btc']
        
        # 1. Momentum e Tendência
        print(f"   🚀 Momentum e Tendência (1h):")
        
        momentum_status = "FORTE ALTA" if momentum_1h > 2 else "ALTA" if momentum_1h > 0.5 else "NEUTRA" if momentum_1h > -0.5 else "BAIXA"
        print(f"      • Momentum: {momentum_status} ({momentum_1h:+.2f}%)")
        
        log_return_status = "ALTA" if log_return_1h > 0.5 else "POSITIVA" if log_return_1h > 0 else "NEGATIVA"
        print(f"      • Retorno: {log_return_status} ({log_return_1h:+.2f}%)")
        
        print(f"")
        
        # 2. RSI
        print(f"   📈 Indicadores Técnicos:")
        
        if rsi_1h < 30:
            rsi_status = "SOBREVENDA (forte)"
        elif rsi_1h < 40:
            rsi_status = "SOBREVENDA"
        elif rsi_1h < 60:
            rsi_status = "NEUTRO"
        elif rsi_1h < 70:
            rsi_status = "COMPRA"
        else:
            rsi_status = "SOBRECOMPRA"
        print(f"      • RSI 1h: {rsi_1h:.1f} ({rsi_status})")
        
        # Bandas de Bollinger
        if bb_position_1h < 0.2:
            bb_status = "BANDA INFERIOR (potencial reversão)"
        elif bb_position_1h < 0.4:
            bb_status = "ABAIXO DA MÉDIA"
        elif bb_position_1h < 0.6:
            bb_status = "ZONA NEUTRA"
        elif bb_position_1h < 0.8:
            bb_status = "ACIMA DA MÉDIA"
        else:
            bb_status = "BANDA SUPERIOR (alta pressão)"
        print(f"      • Bollinger: {bb_position_1h:.2f} ({bb_status})")
        
        print(f"")
        
        # 3. Volatilidade e Volume
        print(f"   ⚡ Volatilidade e Volume:")
        
        vol_status = "MUITO ALTA" if volatility_1h > 0.05 else "ALTA" if volatility_1h > 0.03 else "MÉDIA" if volatility_1h > 0.02 else "BAIXA"
        print(f"      • Volatilidade 1h: {vol_status} ({volatility_1h:.4f})")
        
        vol_change_status = "FORTE AUMENTO" if vol_change_1h > 50 else "AUMENTO" if vol_change_1h > 10 else "ESTÁVEL" if vol_change_1h > -10 else "QUEDA"
        print(f"      • Mudança Volume: {vol_change_status} ({vol_change_1h:+.1f}%)")
        
        print(f"")
        
        # 4. Correlações BTC/ETH
        print(f"   🔗 Correlações com BTC:")
        
        btc_trend = "ALTA" if btc_log_return > 0.5 else "POSITIVA" if btc_log_return > 0 else "NEGATIVA"
        print(f"      • BTC tendência: {btc_trend} ({btc_log_return:+.2f}%)")
        
        if relative_strength_btc > 1:
            rel_strength_status = "FORTE OUTPERFORMANCE"
        elif relative_strength_btc > 0.2:
            rel_strength_status = "OUTPERFORMANCE"
        elif relative_strength_btc > -0.2:
            rel_strength_status = "SIMILAR"
        else:
            rel_strength_status = "UNDERPERFORMANCE"
        print(f"      • Força relativa vs BTC: {rel_strength_status} ({relative_strength_btc:+.2f}%)")
        
        beta_status = "ALTA" if beta_btc > 1.5 else "MODERADA" if beta_btc > 1.0 else "BAIXA"
        print(f"      • Beta BTC: {beta_status} ({beta_btc:.2f})")
        
        print(f"")
        
        # 5. Features chave para modelo (top 5 mais relevantes)
        print(f"   🎯 Features Chave:")
        print(f"      • momentum_1h: {momentum_1h:+.2f}%")
        print(f"      • rsi_1h: {rsi_1h:.1f}")
        print(f"      • bb_position_1h: {bb_position_1h:.2f}")
        print(f"      • relative_strength_btc: {relative_strength_btc:+.2f}%")
        print(f"      • volatility_1h: {volatility_1h:.4f}")
        print(f"   {'─'*60}\n")
    
    def open_position(self, symbol, price, prob, atr):
        """Abre uma posição de compra com TP/SL adaptativos."""
        if symbol in self.positions:
            print(f"{symbol}: Posição já aberta")
            return
        
        # Calcular tamanho da posição
        position_value = self.capital * POSITION_SIZE
        size = position_value / price
        
        # Calcular TP e SL ADAPTATIVOS baseados em ATR
        tp_price = price + (ATR_MULTIPLIER_TP * atr)
        sl_price = price - (ATR_MULTIPLIER_SL * atr)
        
        # Calcular percentuais para display
        tp_pct = ((tp_price / price) - 1) * 100
        sl_pct = ((sl_price / price) - 1) * 100
        
        # Registrar posição
        self.positions[symbol] = {
            'entry_price': price,
            'entry_time': datetime.now(),
            'size': size,
            'tp': tp_price,
            'sl': sl_price,
            'atr': atr,
            'entry_prob': prob
        }
        
        print(f"   🟢 COMPRA: {symbol}")
        print(f"      Preço: ${price:.4f}")
        print(f"      Tamanho: {size:.4f}")
        print(f"      ATR: ${atr:.4f}")
        print(f"      TP: ${tp_price:.4f} (+{tp_pct:.2f}%)")
        print(f"      SL: ${sl_price:.4f} ({sl_pct:.2f}%)")
        print(f"      Probabilidade: {prob*100:.2f}%")
        
        if not self.dry_run:
            # TODO: executar ordem real na exchange
            # self.exchange.create_market_buy_order(symbol, size)
            pass
    
    def close_position(self, symbol, price, reason):
        """Fecha uma posição."""
        if symbol not in self.positions:
            return
        
        pos = self.positions[symbol]
        profit_pct = (price / pos['entry_price'] - 1) * 100
        profit_usd = (price - pos['entry_price']) * pos['size']
        holding_time = (datetime.now() - pos['entry_time']).total_seconds() / 3600
        
        # atualiza capital
        self.capital += profit_usd
        
        # registra trade
        trade = {
            'symbol': symbol,
            'entry_time': pos['entry_time'],
            'exit_time': datetime.now(),
            'entry_price': pos['entry_price'],
            'exit_price': price,
            'profit_pct': profit_pct,
            'profit_usd': profit_usd,
            'holding_hours': holding_time,
            'exit_reason': reason
        }
        self.trade_history.append(trade)
        
        print(f"   🔴 VENDA: {symbol} ({reason})")
        print(f"      Preço: ${price:.4f}")
        print(f"      P&L: {profit_pct:+.2f}% (${profit_usd:+.2f})")
        print(f"      Holding: {holding_time:.1f}h")
        print(f"      Capital: ${self.capital:,.2f}")
        
        # remove posição
        del self.positions[symbol]
        
        if not self.dry_run:
            # TODO: executar ordem real na exchange
            # self.exchange.create_market_sell_order(symbol, pos['size'])
            pass
    
    def check_positions(self):
        """Verifica posições abertas para TP/SL."""
        for symbol in list(self.positions.keys()):
            features, price, atr = self.get_features(symbol)
            if price is None:
                continue
            
            pos = self.positions[symbol]
            
            # Verificar Take Profit
            if price >= pos['tp']:
                self.close_position(symbol, price, 'TAKE PROFIT')
            # Verificar Stop Loss
            elif price <= pos['sl']:
                self.close_position(symbol, price, 'STOP LOSS')
    
    def scan_opportunities(self):
        """Escaneia oportunidades de entrada."""
        # Verificar limite de posições
        if len(self.positions) >= MAX_POSITIONS:
            print(f"   Máximo de {MAX_POSITIONS} posições atingido")
            return
        
        # Escanear cada par
        for symbol in self.pairs:
            # Pular se já temos posição
            if symbol in self.positions:
                continue
            
            # Obter features, preço e ATR
            features, price, atr = self.get_features(symbol)
            if features is None or price is None or atr is None:
                continue
            
            # Verificar se há NaN nas features
            if features.isna().any():
                print(f"{symbol}: Features contêm NaN (dados insuficientes)")
                continue
            
            # Predição
            prob = self.predict(features)
            
            print(f"{symbol}: ${price:.4f} | Prob: {prob*100:.2f}%", end="")
            
            # sinal de compra?
            if prob >= self.threshold:
                print(f" ⬆️ SINAL!")
                # EXPLICAR O SINAL
                self.explain_signal(features, prob)
                self.open_position(symbol, price, prob, atr)
            else:
                print(f" ➡️ Hold")
    
    def print_status(self):
        """Imprime status"""
        print(f"\n{'='*70}")
        print(f"STATUS - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'='*70}")
        print(f"\n Capital: ${self.capital:,.2f} ({(self.capital/self.initial_capital - 1)*100:+.2f}%)")
        print(f" Posições abertas: {len(self.positions)}/{MAX_POSITIONS}")
        print(f" Trades realizados: {len(self.trade_history)}")
        
        if self.positions:
            print(f"\nPosições:")
            for symbol, pos in self.positions.items():
                holding = (datetime.now() - pos['entry_time']).total_seconds() / 3600
                print(f"   • {symbol}: ${pos['entry_price']:.4f} (holding: {holding:.1f}h)")
        
        if self.trade_history:
            trades_df = pd.DataFrame(self.trade_history)
            win_rate = (trades_df['profit_pct'] > 0).sum() / len(trades_df) * 100
            avg_profit = trades_df['profit_pct'].mean()
            print(f"\nPerformance:")
            print(f"   • Win rate: {win_rate:.1f}%")
            print(f"   • Profit médio: {avg_profit:+.2f}%")
    
    def run(self):
        """Loop principal"""
        print(f"\nBot iniciado em modo {'DRY RUN' if self.dry_run else 'LIVE'}")
        print(f"   Pressione Ctrl+C para parar\n")
        
        try:
            while True:
                # Verificar posições abertas
                if self.positions:
                    print(f"\nVerificando {len(self.positions)} posições...")
                    self.check_positions()
                
                # Escanear novas oportunidades
                print(f"\nEscaneando oportunidades...")
                self.scan_opportunities()
                
                # Imprimir status
                self.print_status()
                
                # Aguardar próximo ciclo
                print(f"\nAguardando {CHECK_INTERVAL}s...\n")
                time.sleep(CHECK_INTERVAL)
                
        except KeyboardInterrupt:
            print(f"\n\nBot interrompido pelo usuário")
            self.print_status()

print("  • explain_signal: explica motivo da probabilidade alta")
print("  • scan_opportunities: integrado com explain_signal")
print("  • Detalhes das features agora são exibidos quando detectar sinal")

## Modo Contínuo (Padrão)
* Bot executa em loop infinito
* Verifica posições abertas a cada 60s
* Escaneia oportunidades de entrada
* Exibe status após cada ciclo

## O que acontece durante a execução:

**A cada 60 segundos:**
1. Verifica posições abertas (TP/SL)
2. Escaneia novos sinais de compra
3. Exibe status (capital, trades, posições)
4. Aguarda próximo ciclo

In [0]:
# # cria instância
# bot = TradingBot(
#     exchange_id=EXCHANGE,
#     model=model,
#     pairs=TRADING_PAIRS,
#     threshold=THRESHOLD,
#     dry_run=DRY_RUN,
#     api_key=API_KEY,
#     api_secret=API_SECRET,
#     api_passphrase=API_PASSPHRASE,
#     initial_capital=INITIAL_CAPITAL
# )
# print(f"\n{'='*70}")
# print(f"EXECUTANDO...")
# print(f"{'='*70}")
# print(f"\nO bot verificará posições e oportunidades a cada {CHECK_INTERVAL} segundos")
# print(f"{'='*70}\n")
# # loop infinito até interrupção manual
# bot.run()

# =============================================================================
# MODO TESTE (3 iterações) - Descomente para testes rápidos
# =============================================================================

# cria instância
bot = TradingBot(
    exchange_id=EXCHANGE,
    model=model,
    pairs=TRADING_PAIRS,
    threshold=THRESHOLD,
    dry_run=DRY_RUN,
    api_key=API_KEY,
    api_secret=API_SECRET,
    api_passphrase=API_PASSPHRASE,
    initial_capital=INITIAL_CAPITAL
)

# Rodar 3 iterações de teste
print(f"\n{'='*70}")
print("MODO TESTE: Executando 3 iterações")
print(f"{'='*70}\n")

for iteration in range(1, 4):
    print(f"\n{'='*70}")
    print(f"ITERAÇÃO {iteration}/3")
    print(f"{'='*70}")
    
    # 1. Verificar posições abertas
    if bot.positions:
        print(f"\nVerificando {len(bot.positions)} posições...")
        bot.check_positions()
    
    # 2. Escanear novas oportunidades
    print(f"\nEscaneando oportunidades...")
    bot.scan_opportunities()
    
    # 3. Imprimir status
    bot.print_status()
    
    # 4. Aguardar próxima iteração (exceto na última)
    if iteration < 3:
        print(f"Aguardando {CHECK_INTERVAL}s até próxima iteração...\n")
        time.sleep(CHECK_INTERVAL)

print(f"\n{'='*70}")
print("TESTE CONCLUÍDO")
print(f"{'='*70}")
print(f"\nResumo Final:")
print(f"   • Capital final: ${bot.capital:,.2f}")
print(f"   • Retorno: {(bot.capital/bot.initial_capital - 1)*100:+.2f}%")
print(f"   • Trades executados: {len(bot.trade_history)}")
print(f"   • Posições abertas: {len(bot.positions)}")

if bot.trade_history:
    trades_df = pd.DataFrame(bot.trade_history)
    win_rate = (trades_df['profit_pct'] > 0).sum() / len(trades_df) * 100
    print(f"   • Win rate: {win_rate:.1f}%")
    print(f"   • Profit médio: {trades_df['profit_pct'].mean():+.2f}%")